In [14]:
#Gradients of loss with respect to weights
import numpy as np

dvalues=np.array([[1.,1.,1.],
                  [2., 2., 2.],
                  [3., 3., 3.]])
#We have 3 sets of inputs-samples
inputs=np.array([[1,2,3,2.5],
                 [2., 5., -1., 2],
                 [-1.5, 2.7,3.3, -0.8]])
#sum weights of given input and multiply by the passed-in gradient for this neuron
dweights=np.dot(inputs.T,dvalues)
print(dweights)

[[ 0.5  0.5  0.5]
 [20.1 20.1 20.1]
 [10.9 10.9 10.9]
 [ 4.1  4.1  4.1]]


In [15]:
#Gradients of the loss with respect to biases

dbiases=np.sum(dvalues, axis=0, keepdims=True)
print(dbiases)

[[6. 6. 6.]]


In [16]:
# Gradients of the loss with respect to inputs

dvalues=np.array([[1.,1.,1.],
                  [2., 2., 2.],
                  [3., 3., 3.]])

weights=np.array([[0.2,0.8,-0.5,1],
                  [0.5,-0.91,0.26, -0.5],
                  [-0.26, -0.27, 0.17, 0.87]]).T
dinputs=np.dot(dvalues, weights.T)

print(dinputs)


[[ 0.44 -0.38 -0.07  1.37]
 [ 0.88 -0.76 -0.14  2.74]
 [ 1.32 -1.14 -0.21  4.11]]


# Code till now


In [17]:
#import packages
import numpy as np
import nnfs
from nnfs.datasets import spiral_data
nnfs.init()
import matplotlib.pyplot as plt

# Adding the "Backward" Method in the Layer_Dense class, Relu activation and categorical cross entropy class


In [18]:

#Dense layer

class Layer_dense:
    #Layer intialization
    def __init__(self, n_inputs, n_neurons):
        self.weights=0.01*np.random.randn(n_inputs, n_neurons)
        self.biases=np.zeros((1,n_neurons))

    #Forward pass
    def forward(self, inputs):
        #Calculate output values from inputs, weights, biases
        self.output=np.dot(inputs, self.weights)+self.biases

    def backward(self, dvalues):
        #gradients on parameters
        self.dweights=np.dot(self.inputs.T, dvalues)
        self.dbiases=np.sum(dvalues, axis=0,keepdims=True)

        #gradient on values
        self.dinputs=np.dot(dvalues, self.weights.T)

In [ ]:
# Relu activation 
class Activation_Relu:
    #forward pass
    def forward(self,inputs):
        #Calculate output values from input
        self.inputs=inputs
        self.output=np.maximum(0,inputs)

        #Backward pass

    def backward(self, dvalues):
        #since we need to modify the original variable 
        #Make a copy of the values first

        self.dinputs=dvalues.copy()
        self.dinputs[self.inputs <= 0] =0


In [20]:
#Softmax activation

class Activation_Softmax:
    # Forward Pass
    def forward(self, inputs):
        #Get unnormalized probabilities
        exp_values=np.exp(inputs-np.max(inputs, axis=1, keepdims=True))
        #Normalize them for each sample
        probabilities=exp_values/np.sum(exp_values, axis=1, keepdims=True)
        self.output=probabilities

In [21]:
# Implementing the loss class

class Loss:
    # Calculates the data and regularization losses
    def calculate(self, output,y):
        #Calculate sample losses
        sample_losses=self.forward(output,y)
        #Calculate mean loss
        data_loss=np.mean(sample_losses)
        #return Loss
        return data_loss

#we implement the class loss just for the sake of simplicity
#So we can also see weight values

In [22]:
# Implemnting the categorical cross entropy class

class Loss_CategoricalCrossentropy(Loss):

    #Backward [ass
    def backward(self, dvalues, y_true):
        # Number of samples
        samples=len(dvalues)
        #number of labels in every sample
        labels=len(dvalues[0])
        # If labels are sparse, turn them into one-hot vector
        if len(y_true.shape)==1:
            y_true=np.eye(labels)[y_true]
        # calculate gradient
        self.dinputs=-y_true/dvalues
        #normalize gradient
        self.dinputs=self.dinputs/samples
    #forward pass 
    def forward(self,y_pred,y_true):
        samples=len(y_pred)
        #Clip data to prevent divisions by 0
        #Clip both sides to not drag mean towards any value
        y_pred_clipped=np.clip(y_pred, 1e-7,1-1e-7)
        if len(y_true.shape)==1:
            correct_confidences=y_pred_clipped[
                range(samples),
                y_true
            ]
            #Mask values -only for one-hot encoded labels
        elif len(y_true.shape)==2:
            correct_confidences = np.sum(
                y_pred_clipped*y_true,
                axis=1
            )
        #losses
        negative_log_likelihoods=-np.log(correct_confidences)
        return negative_log_likelihoods


# Softmax Classifier-combined Softmax activation and cross-entropy loss for faster backward step

In [23]:
class Activation_Softmax_Loss_categoricalCrossentropy:
    #creates activation and loss function objects
    def __init__(self):
        self.activation=Activation_Softmax()
        self.loss=Loss_CategoricalCrossentropy()

    #forward pass
    def forward(self, inputs, y_true):
        #Output layer's activation function
        self.activation.forward(inputs)
        # Set the output
        self.output=self.activation.output
        # Calculate and return loss values
        return self.loss.calculate(self.output, y_true)
    
        #Backward pass
    def backward(self, dvalues, y_true):
        samples=len(dvalues)
        if len(y_true.shape)==2:
            y_true=np.argmax(y_true, axis=1)
        self.dinputs=dvalues.copy()
        #calculate the gradient
        self.dinputs[range(samples), y_true]-=1
        #Normalize gradient
        self.dinputs=self.dinputs/samples



In [24]:
#Lets see how our code works in action

softmax_outputs=np.array([[0.7,0.1,0.2],
                          [0.1,0.5,0.4],
                          [0.02, 0.9, 0.08]])
class_targets=np.array([0, 1,1])
softmax_loss=Activation_Softmax_Loss_categoricalCrossentropy()
softmax_loss.backward(softmax_outputs, class_targets)
dvalues1=softmax_loss.dinputs
print("Gradients:combined loss and activation:")
print(dvalues1)


Gradients:combined loss and activation:
[[-0.1         0.03333333  0.06666667]
 [ 0.03333333 -0.16666667  0.13333333]
 [ 0.00666667 -0.03333333  0.02666667]]
